# Course 4: Large Language Models
## Session 12: Capstone: the smallest useful Bangla LLM

**Goal:** distil a larger model into a smaller one, compare it with training the small model alone, and fill in a model card.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

### 1. A teacher with plenty of data, a student with little
The teacher is trained on 1,200 sentences. The student is four times narrower and has only 60 sentences of its own. This mirrors the real case: a large multilingual model has seen far more Bangla than you can collect.

In [ ]:
import math, numpy as np
teacher_cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=64)
student_cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=16)
teacher = train_lm(teacher_cfg, steps=600)
small_ids = train_ids[:60 * 6]                                      # the student's own 60 sentences
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"teacher: {count(teacher)} parameters, validation loss {val_loss(teacher):.3f}")

### 2. Three students
1. Trained alone on its 60 sentences.
2. Distilled: on the same 60 sentences, it also matches the teacher's full next-word distribution (soft targets), which says far more than the one correct word.
3. Distilled on text the teacher writes: unlimited synthetic data, no extra real data.

In [ ]:
def train_student(mode, steps=800, alpha=0.9, temperature=1.0, seed=0):
    torch.manual_seed(seed); s = GPT(student_cfg); opt = torch.optim.AdamW(s.parameters(), lr=3e-3)
    for _ in range(steps):
        if mode == "synthetic":
            with torch.no_grad(): x = teacher.generate(get_batch(train_ids, 6, 32)[0], 25, top_p=1.0)[:, 6:]; x, y = x[:, :-1].contiguous(), x[:, 1:].contiguous()
        else:
            x, y = get_batch(small_ids, 24, 32)
        logits, ce = s(x, y)
        if mode == "alone": loss = ce
        else:
            with torch.no_grad(): t_logits = teacher(x)[0]
            kd = F.kl_div(torch.log_softmax(logits / temperature, -1), torch.softmax(t_logits / temperature, -1), reduction="batchmean") * temperature ** 2 / x.size(1)
            loss = alpha * kd + (1 - alpha) * ce
        opt.zero_grad(); loss.backward(); opt.step()
    return s

floor = math.log(len(ALL_SENTENCES)) / 6
results = {}
for mode in ["alone", "distilled", "synthetic"]:
    runs = [train_student(mode, seed=sd) for sd in range(3)]
    losses = [val_loss(r) for r in runs]; valid = np.mean([sum(map(is_valid, sample_sentences(r))) / 200 for r in runs])
    results[mode] = (np.mean(losses), np.std(losses), valid)
    print(f"student, {mode:10s} validation loss {np.mean(losses):.3f} ± {np.std(losses):.3f} | valid sentences {valid:.2f}")
print(f"student size: {count(runs[0])} parameters ({count(teacher) / count(runs[0]):.1f} times smaller than the teacher) | best possible loss {floor:.3f}")
assert results["distilled"][0] < results["alone"][0] and results["synthetic"][0] < results["alone"][0]

### 3. How small can it go?
Shrink the student until it stops being useful. 'Useful' needs a definition before you start: here, at least 90% valid sentences.

In [ ]:
print("d_model   parameters   validation loss   valid sentences")
for d in [4, 8, 16, 32]:
    student_cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=d)
    s = train_student("synthetic", steps=600)
    v = sum(map(is_valid, sample_sentences(s))) / 200
    print(f"{d:7d}   {count(s):10d}   {val_loss(s):15.3f}   {v:15.2f}" + ("   <- useful by our definition" if v >= 0.9 else ""))

### 4. The capstone on a real model (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU, model downloads and your tokenizer and data from earlier sessions.
# Teacher: an open Bangla-capable model (for example a TituLLMs checkpoint, or a multilingual model fine-tuned in session 5).
# Student: a model of 10 to 50 million parameters with your Bangla tokenizer from session 2.
# Recipe:
#   1. Data: clean Bangla text plus text generated by the teacher from simple prompts (the TinyStories idea).
#   2. Pretrain the student on the data; then distil with soft targets on a subset if the tokenizers match,
#      or on teacher-written text if they do not (soft targets need the same vocabulary).
#   3. Evaluate on: your 100-question set from session 11, perplexity on held-out text, and two BanglaBERT/BLUB-style tasks after fine-tuning.
#   4. Compare with the published TituLLMs and BanglaBERT numbers, and state every difference in setup that makes the comparison unfair.

### 5. Model card (TODO: fill in for your capstone model)

| Field | Your entry |
|---|---|
| Name, size in parameters and in megabytes | |
| What it is for, in one sentence | |
| What it must not be used for | |
| Training data: sources, size in tokens, licence | |
| Tokenizer: type, vocabulary size, tokens per Bangla word | |
| Teacher model, if any, and its licence | |
| Compute used | |
| Results: each benchmark, with the number of seeds and the interval | |
| Comparison rows: TituLLMs, BanglaBERT, and what differs in the setup | |
| Known failures: three real examples | |
| What you would do with ten times the compute | |

### 6. Presentation (10 minutes)
- 1 slide: the goal and the definition of 'useful'.
- 1 slide: data and model.
- 2 slides: results with error bars, and the size-quality curve from section 3.
- 1 slide: failures, limits and next steps.
- A live demo of 3 prompts chosen by the audience.